# 03 Technical Indicators

Compute technical indicators for Nikkei 225 and build a feature matrix for modeling.

| Section | Content |
|---------|--------|
| 1 | Data loading |
| 2 | Moving averages (SMA / EMA) |
| 3 | Bollinger Bands |
| 4 | RSI |
| 5 | MACD |
| 6 | Volume indicators |
| 7 | ATR (volatility) |
| 8 | Stochastic oscillator |
| 9 | Feature engineering (model-ready) |
| 10 | Save to data/processed/ |

In [ ]:
# ── Step 1: Install packages ───────────────────────────────────────────────
import subprocess, sys

pkgs = ['yfinance', 'pandas-ta', 'scikit-learn>=1.5.0', 'statsmodels>=0.14.0']
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q'] + pkgs,
    capture_output=True, text=True
)
out = result.stdout + result.stderr
print(out.strip() if out.strip() else 'All packages already up to date.')

if 'Successfully installed' in out:
    print('\nPackages upgraded — restarting runtime...')
    import os; os.kill(os.getpid(), 9)

In [ ]:
# ── Step 2: Repository & path setup ───────────────────────────────────────
import os, sys

REPO = '/content/Nikkei_Analysis'

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists(REPO):
        !git clone https://github.com/Takumi-Itokawa-Finance/Nikkei_Analysis.git {REPO}
    else:
        !git -C {REPO} pull
    os.chdir(REPO)

    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DATA = '/content/drive/MyDrive/Nikkei_Analysis/data'
    os.makedirs(DRIVE_DATA, exist_ok=True)
    if not os.path.exists(f'{REPO}/data'):
        os.symlink(DRIVE_DATA, f'{REPO}/data')

    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    print(f'Setup complete.  CWD={os.getcwd()}')
else:
    print('Running in local environment.')

## 1. Data Loading

In [ ]:
import warnings
import numpy as np
import pandas as pd
import pandas_ta as ta
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from src.data.fetcher import fetch_nikkei

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

OUTPUT_FIGS = 'output/figures'
OUTPUT_DATA = 'data/processed'
os.makedirs(OUTPUT_FIGS, exist_ok=True)
os.makedirs(OUTPUT_DATA, exist_ok=True)

In [ ]:
raw = fetch_nikkei(period='5y', interval='1d')

# Flatten MultiIndex columns if present (yfinance >= 0.2)
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)

df = raw[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
df = df.ffill().dropna()

print(f'Shape  : {df.shape}')
print(f'Period : {df.index[0].date()} to {df.index[-1].date()}')
display(df.tail())

## 2. Moving Averages (SMA / EMA)

In [ ]:
sma_periods = [5, 10, 20, 50, 75, 200]
ema_periods = [5, 20, 50]

for p in sma_periods:
    df[f'SMA_{p}'] = ta.sma(df['Close'], length=p)

for p in ema_periods:
    df[f'EMA_{p}'] = ta.ema(df['Close'], length=p)

print('Moving averages computed.')
display(df[[f'SMA_{p}' for p in sma_periods] + [f'EMA_{p}' for p in ema_periods]].tail())

In [ ]:
plot_df = df.last('1y')

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(plot_df.index, plot_df['Close'], label='Close', color='black', linewidth=1.2)
for p, color in zip([20, 50, 200], ['steelblue', 'orange', 'red']):
    ax.plot(plot_df.index, plot_df[f'SMA_{p}'], label=f'SMA {p}', linewidth=1, color=color)
ax.set_title('Nikkei 225 — Close & Moving Averages (1Y)')
ax.set_ylabel('Price (JPY)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.tight_layout()
plt.savefig(f'{OUTPUT_FIGS}/03_moving_averages.png', dpi=150)
plt.show()

## 3. Bollinger Bands

In [ ]:
bb = ta.bbands(df['Close'], length=20, std=2)
df['BB_lower'] = bb['BBL_20_2.0']
df['BB_mid']   = bb['BBM_20_2.0']
df['BB_upper'] = bb['BBU_20_2.0']
df['BB_width'] = bb['BBB_20_2.0']   # bandwidth = (upper - lower) / mid
df['BB_pct']   = bb['BBP_20_2.0']   # %B = (close - lower) / (upper - lower)

display(df[['Close', 'BB_lower', 'BB_mid', 'BB_upper', 'BB_width', 'BB_pct']].tail())

In [ ]:
plot_df = df.last('1y')

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(plot_df.index, plot_df['Close'], color='black', linewidth=1.2, label='Close')
axes[0].plot(plot_df.index, plot_df['BB_upper'], color='steelblue', linewidth=0.8, label='Upper')
axes[0].plot(plot_df.index, plot_df['BB_mid'],   color='grey',      linewidth=0.8, linestyle='--', label='Mid')
axes[0].plot(plot_df.index, plot_df['BB_lower'], color='steelblue', linewidth=0.8, label='Lower')
axes[0].fill_between(plot_df.index, plot_df['BB_lower'], plot_df['BB_upper'], alpha=0.1, color='steelblue')
axes[0].set_title('Bollinger Bands (20, 2)')
axes[0].set_ylabel('Price (JPY)')
axes[0].legend()

axes[1].plot(plot_df.index, plot_df['BB_pct'], color='purple', linewidth=1)
axes[1].axhline(1.0, color='steelblue', linewidth=0.8, linestyle='--', label='Overbought')
axes[1].axhline(0.0, color='tomato',    linewidth=0.8, linestyle='--', label='Oversold')
axes[1].set_title('%B (Bollinger Band Position)')
axes[1].set_ylabel('%B')
axes[1].legend()
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

plt.tight_layout()
plt.savefig(f'{OUTPUT_FIGS}/03_bollinger_bands.png', dpi=150)
plt.show()

## 4. RSI (Relative Strength Index)

In [ ]:
df['RSI_14'] = ta.rsi(df['Close'], length=14)
df['RSI_9']  = ta.rsi(df['Close'], length=9)

display(df[['Close', 'RSI_14', 'RSI_9']].tail())

In [ ]:
plot_df = df.last('1y')

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(plot_df.index, plot_df['Close'], color='black', linewidth=1.2)
axes[0].set_title('Nikkei 225 Close')
axes[0].set_ylabel('Price (JPY)')

axes[1].plot(plot_df.index, plot_df['RSI_14'], color='steelblue', linewidth=1, label='RSI 14')
axes[1].plot(plot_df.index, plot_df['RSI_9'],  color='orange',    linewidth=1, label='RSI 9', alpha=0.7)
axes[1].axhline(70, color='tomato',    linewidth=0.8, linestyle='--', label='Overbought (70)')
axes[1].axhline(30, color='steelblue', linewidth=0.8, linestyle='--', label='Oversold (30)')
axes[1].axhline(50, color='grey',      linewidth=0.6, linestyle=':')
axes[1].fill_between(plot_df.index, plot_df['RSI_14'], 70,
                     where=(plot_df['RSI_14'] >= 70), alpha=0.2, color='tomato')
axes[1].fill_between(plot_df.index, plot_df['RSI_14'], 30,
                     where=(plot_df['RSI_14'] <= 30), alpha=0.2, color='steelblue')
axes[1].set_title('RSI')
axes[1].set_ylabel('RSI')
axes[1].set_ylim(0, 100)
axes[1].legend()
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

plt.tight_layout()
plt.savefig(f'{OUTPUT_FIGS}/03_rsi.png', dpi=150)
plt.show()

## 5. MACD

In [ ]:
macd = ta.macd(df['Close'], fast=12, slow=26, signal=9)
df['MACD']        = macd['MACD_12_26_9']
df['MACD_signal'] = macd['MACDs_12_26_9']
df['MACD_hist']   = macd['MACDh_12_26_9']

display(df[['Close', 'MACD', 'MACD_signal', 'MACD_hist']].tail())

In [ ]:
plot_df = df.last('1y')

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(plot_df.index, plot_df['Close'], color='black', linewidth=1.2)
axes[0].set_title('Nikkei 225 Close')
axes[0].set_ylabel('Price (JPY)')

axes[1].plot(plot_df.index, plot_df['MACD'],        color='steelblue', linewidth=1,   label='MACD')
axes[1].plot(plot_df.index, plot_df['MACD_signal'], color='orange',    linewidth=1,   label='Signal')
axes[1].bar(plot_df.index,  plot_df['MACD_hist'],
            color=['steelblue' if v >= 0 else 'tomato' for v in plot_df['MACD_hist']],
            alpha=0.5, label='Histogram')
axes[1].axhline(0, color='black', linewidth=0.6)
axes[1].set_title('MACD (12, 26, 9)')
axes[1].legend()
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

plt.tight_layout()
plt.savefig(f'{OUTPUT_FIGS}/03_macd.png', dpi=150)
plt.show()

## 6. Volume Indicators

In [ ]:
df['Volume_SMA20'] = ta.sma(df['Volume'], length=20)
df['Volume_ratio'] = df['Volume'] / df['Volume_SMA20']  # ratio vs recent average
df['OBV']          = ta.obv(df['Close'], df['Volume'])

display(df[['Close', 'Volume', 'Volume_SMA20', 'Volume_ratio', 'OBV']].tail())

In [ ]:
plot_df = df.last('1y')

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(plot_df.index, plot_df['Close'], color='black', linewidth=1.2)
axes[0].set_title('Nikkei 225 Close')
axes[0].set_ylabel('Price (JPY)')

axes[1].bar(plot_df.index, plot_df['Volume'], color='steelblue', alpha=0.6, label='Volume')
axes[1].plot(plot_df.index, plot_df['Volume_SMA20'], color='orange', linewidth=1, label='SMA 20')
axes[1].set_title('Volume')
axes[1].set_ylabel('Volume')
axes[1].legend()

axes[2].plot(plot_df.index, plot_df['OBV'], color='purple', linewidth=1)
axes[2].set_title('On-Balance Volume (OBV)')
axes[2].set_ylabel('OBV')
axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

plt.tight_layout()
plt.savefig(f'{OUTPUT_FIGS}/03_volume.png', dpi=150)
plt.show()

## 7. ATR — Average True Range (Volatility)

In [ ]:
df['ATR_14'] = ta.atr(df['High'], df['Low'], df['Close'], length=14)
df['ATR_pct'] = df['ATR_14'] / df['Close'] * 100  # normalised as % of price

display(df[['Close', 'ATR_14', 'ATR_pct']].tail())

In [ ]:
plot_df = df.last('1y')

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].plot(plot_df.index, plot_df['Close'], color='black', linewidth=1.2)
axes[0].set_title('Nikkei 225 Close')
axes[0].set_ylabel('Price (JPY)')

axes[1].plot(plot_df.index, plot_df['ATR_pct'], color='darkorange', linewidth=1)
axes[1].fill_between(plot_df.index, plot_df['ATR_pct'], alpha=0.2, color='orange')
axes[1].set_title('ATR % (14-day, normalised by Close)')
axes[1].set_ylabel('ATR %')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

plt.tight_layout()
plt.savefig(f'{OUTPUT_FIGS}/03_atr.png', dpi=150)
plt.show()

## 8. Stochastic Oscillator

In [ ]:
stoch = ta.stoch(df['High'], df['Low'], df['Close'], k=14, d=3)
df['Stoch_K'] = stoch['STOCHk_14_3_3']
df['Stoch_D'] = stoch['STOCHd_14_3_3']

display(df[['Close', 'Stoch_K', 'Stoch_D']].tail())

In [ ]:
plot_df = df.last('1y')

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(plot_df.index, plot_df['Close'], color='black', linewidth=1.2)
axes[0].set_title('Nikkei 225 Close')
axes[0].set_ylabel('Price (JPY)')

axes[1].plot(plot_df.index, plot_df['Stoch_K'], color='steelblue', linewidth=1, label='%K')
axes[1].plot(plot_df.index, plot_df['Stoch_D'], color='orange',    linewidth=1, label='%D')
axes[1].axhline(80, color='tomato',    linewidth=0.8, linestyle='--', label='Overbought (80)')
axes[1].axhline(20, color='steelblue', linewidth=0.8, linestyle='--', label='Oversold (20)')
axes[1].set_title('Stochastic Oscillator (14, 3)')
axes[1].set_ylabel('%K / %D')
axes[1].set_ylim(0, 100)
axes[1].legend()
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

plt.tight_layout()
plt.savefig(f'{OUTPUT_FIGS}/03_stochastic.png', dpi=150)
plt.show()

## 9. Feature Engineering — Model-Ready Matrix

Convert raw indicators into features suitable for prediction.

Key principle: **features must be available at time t to predict t+1**.
All features are derived from same-day OHLCV — no look-ahead bias.

Raw price levels → normalised / stationary alternatives:

| Raw | Model-ready |
|-----|-------------|
| SMA_20 | Close / SMA_20 - 1 (price relative to MA) |
| BB_upper / lower | BB_pct (already 0-1 scaled) |
| Volume | Volume_ratio (vs 20-day average) |
| ATR | ATR_pct (% of price) |
| MACD | MACD_hist (already a difference) |
| RSI | as-is (bounded 0-100) |
| Stoch | as-is (bounded 0-100) |

In [ ]:
feat = pd.DataFrame(index=df.index)

# Target: next-day log return
feat['target_return'] = np.log(df['Close'] / df['Close'].shift(1)).shift(-1)
feat['target_close']  = df['Close'].shift(-1)   # raw next-day close (for inverse transform)

# --- Trend: price relative to moving averages ---
for p in [5, 10, 20, 50, 75, 200]:
    feat[f'close_vs_sma{p}'] = df['Close'] / df[f'SMA_{p}'] - 1

for p in [5, 20, 50]:
    feat[f'close_vs_ema{p}'] = df['Close'] / df[f'EMA_{p}'] - 1

# MA crossover: SMA5/SMA20, SMA20/SMA50, SMA50/SMA200
feat['sma5_vs_sma20']   = df['SMA_5']  / df['SMA_20']  - 1
feat['sma20_vs_sma50']  = df['SMA_20'] / df['SMA_50']  - 1
feat['sma50_vs_sma200'] = df['SMA_50'] / df['SMA_200'] - 1

# --- Volatility ---
feat['BB_pct']   = df['BB_pct']
feat['BB_width'] = df['BB_width']
feat['ATR_pct']  = df['ATR_pct']

# --- Momentum ---
feat['RSI_14']    = df['RSI_14']
feat['RSI_9']     = df['RSI_9']
feat['MACD_hist'] = df['MACD_hist']
feat['Stoch_K']   = df['Stoch_K']
feat['Stoch_D']   = df['Stoch_D']
feat['Stoch_KD']  = df['Stoch_K'] - df['Stoch_D']  # crossover signal

# --- Returns (past N days) ---
for d in [1, 2, 3, 5, 10]:
    feat[f'return_{d}d'] = np.log(df['Close'] / df['Close'].shift(d))

# --- Volume ---
feat['volume_ratio'] = df['Volume_ratio']
feat['obv_change']   = df['OBV'].pct_change()

# Drop rows with NaN (from indicator warm-up periods)
feat = feat.dropna()

print(f'Feature matrix shape : {feat.shape}')
print(f'Period               : {feat.index[0].date()} to {feat.index[-1].date()}')
print(f'\nFeatures ({len(feat.columns) - 2}):')
print([c for c in feat.columns if c not in ['target_return', 'target_close']])

In [ ]:
display(feat.describe().T)

In [ ]:
# Correlation of each technical feature with next-day return
feature_cols = [c for c in feat.columns if c not in ['target_return', 'target_close']]
corr_target = feat[feature_cols].corrwith(feat['target_return']).sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['steelblue' if v >= 0 else 'tomato' for v in corr_target]
corr_target.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Technical Feature Correlation with Next-Day Return')
ax.set_xlabel('Pearson r')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(f'{OUTPUT_FIGS}/03_feature_corr.png', dpi=150)
plt.show()

print('\nTop 10 by absolute correlation:')
display(corr_target.abs().sort_values(ascending=False).head(10).to_frame('abs_corr'))

## 10. Save to data/processed/

In [ ]:
save_path = f'{OUTPUT_DATA}/technical_features.csv'
feat.to_csv(save_path)
print(f'Saved: {save_path}  shape={feat.shape}')
print('\nNext: 04_modeling.ipynb — load this file and train prediction models.')